# 基于 BERT 动态词嵌入 + 手写 4-Head Transformer Encoder 的电影评论情感分析

本 Notebook 演示如何将预训练 **BERT (bert-base-uncased)** 仅作为**冻结的动态词嵌入特征提取器 (Contextual Feature Extractor)**，并将其输出的高阶上下文向量输入到我们**手写的 4 头自注意力 Transformer Encoder** 中进行情感分类。

### 🌟 本架构的核心亮点：
1. **BERT 动态词嵌入 (Feature-based BERT)**：利用预训练 BERT 生成每个词在语境下的 768 维动态向量，彻底解决传统 GloVe/Word2Vec 一词多义的缺陷。
2. **冻结 BERT 权重 (`requires_grad = False`)**：BERT 不参与梯度更新，极大降低显存消耗与训练时间，仅作为高质量特征提取器。
3. **手写 4-Head Transformer Encoder**：下游完整保留自定义的 **4 头自注意力机制（QKV 独立投影矩阵）**、残差连接 (Residual)、层归一化 (LayerNorm) 与前馈网络 (FFN)。
4. **Masked Global Max Pooling**：基于 Attention Mask 进行全局最大池化提取最强情感特征，最后经由全连接层映射为 0/1 二分类。
5. **Early Stopping 早停监控**：在全量 IMDB 数据集上实时监控 `val_loss`，自动保存最佳模型权重 `best_transformer_bert_model.pt`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Change the current working directory
os.chdir('/content/drive/MyDrive/Colab_Data/RNN评论情感分析')

# Verify the current working directory
print(f"Current working directory: {os.getcwd()}")

Mounted at /content/drive
Current working directory: /content/drive/MyDrive/Colab_Data/RNN评论情感分析


In [ ]:
# 如果在 Google Colab 环境下运行，请取消注释并执行挂载与路径切换
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    if os.path.exists('/content/drive/MyDrive/Colab_Data/RNN评论情感分析'):
        os.chdir('/content/drive/MyDrive/Colab_Data/RNN评论情感分析')
        print(f"Current working directory: {os.getcwd()}")
except Exception as e:
    print("本地环境运行，跳过 Google Drive 挂载。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/MyDrive/Colab_Data/RNN评论情感分析


In [ ]:
# 安装 HuggingFace Transformers 库
!pip install -q transformers

## 1. 导入必要的库与全量均衡数据加载

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModel

# 设置随机种子确保可复现性
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def load_balanced_data(file_path, sample_per_class=None, random_seed=42):
    df = pd.read_csv(file_path)
    df_pos = df[df['真实标签'] == 1]
    df_neg = df[df['真实标签'] == 0]
    if sample_per_class is not None:
        pos_sampled = df_pos.sample(n=min(sample_per_class, len(df_pos)), random_state=random_seed)
        neg_sampled = df_neg.sample(n=min(sample_per_class, len(df_neg)), random_state=random_seed)
    else:
        min_len = min(len(df_pos), len(df_neg))
        pos_sampled = df_pos.sample(n=min_len, random_state=random_seed)
        neg_sampled = df_neg.sample(n=min_len, random_state=random_seed)
    return pd.concat([pos_sampled, neg_sampled]).sample(frac=1, random_state=random_seed).reset_index(drop=True)

train_path = 'data/data_train.csv'
test_path = 'data/data_test.csv'

df_raw_train = load_balanced_data(train_path, sample_per_class=None)
df_test = load_balanced_data(test_path, sample_per_class=None)

# 按 85:15 划分 Train 集与 Validation 集
df_train, df_val = train_test_split(df_raw_train, test_size=0.15, random_state=42, stratify=df_raw_train['真实标签'])

print(f"训练集样本数量: {len(df_train)} (正面: {sum(df_train['真实标签']==1)}, 负面: {sum(df_train['真实标签']==0)})")
print(f"验证集样本数量: {len(df_val)}   (正面: {sum(df_val['真实标签']==1)}, 负面: {sum(df_val['真实标签']==0)})")
print(f"测试集样本数量: {len(df_test)}  (正面: {sum(df_test['真实标签']==1)}, 负面: {sum(df_test['真实标签']==0)})")
display(df_train.head())

训练集样本数量: 21250 (正面: 10625, 负面: 10625)
验证集样本数量: 3750   (正面: 1875, 负面: 1875)
测试集样本数量: 25000  (正面: 12500, 负面: 12500)


,影评内容,真实标签
11904,I only watched the first 30 minutes of this an...,0
19929,Its no surprise that Busey later developed a t...,1
15180,This is the second movie I saw for Horrorfest ...,0
12711,"i read the book ""7 years in Tibet"" from Heinri...",0
4295,I don't know what the Oscar voters saw in this...,0


## 2. BERT Tokenizer 分词与 PyTorch Dataset / DataLoader 构建

In [ ]:
def clean_text_for_bert(text):
    if not isinstance(text, str):
        return ""
    # 去除 HTML 换行标签 <br /> 并去除多余空格
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_texts = [clean_text_for_bert(t) for t in df_train['影评内容']]
val_texts = [clean_text_for_bert(t) for t in df_val['影评内容']]
test_texts = [clean_text_for_bert(t) for t in df_test['影评内容']]

# 加载官方预训练 BERT Tokenizer
bert_model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

class BertMovieDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=200):
        self.encodings = tokenizer(
            texts,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label': self.labels[idx]
        }

max_len = 200
batch_size = 64  # 如果显存有限，可调整为 32 或 64

train_dataset = BertMovieDataset(train_texts, df_train['真实标签'].values, tokenizer, max_len)
val_dataset = BertMovieDataset(val_texts, df_val['真实标签'].values, tokenizer, max_len)
test_dataset = BertMovieDataset(test_texts, df_test['真实标签'].values, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train batches: 333 | Val batches: 59 | Test batches: 391


## 3. 构建模型：BERT 词嵌入特征提取器 + 手写 4-Head Transformer Encoder

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    手写 4 头自注意力机制 (Multi-Head Self-Attention)
    显式采用 3 个独立的线性投影矩阵 (W_q, W_k, W_v) 分别变换 Query, Key, Value
    并在多头拼接后通过 out_proj 进行跨头特征融合
    """
    def __init__(self, embed_dim=128, num_heads=4, dropout=0.1):
        super(MultiHeadSelfAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能够被 num_heads 整除"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # 独立的 Q, K, V 投影矩阵
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [batch_size, seq_len, embed_dim]
        batch_size, seq_len, embed_dim = x.size()

        # 映射后拆分 4 头: [batch_size, num_heads, seq_len, head_dim]
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention: Q * K^T / sqrt(head_dim)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 加权求和并拼接 4 个头
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)

        return self.out_proj(context)

class TransformerEncoderBlock(nn.Module):
    """ 单层 Transformer Encoder 模块 """
    def __init__(self, embed_dim=128, num_heads=4, dim_feedforward=256, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()
        self.self_attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sub-layer 1: Attention + 残差 + LayerNorm
        attn_out = self.self_attn(x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Sub-layer 2: FFN + 残差 + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

class BertTransformerClassifier(nn.Module):
    """
    结合 BERT 词嵌入与手写 4-Head Transformer Encoder 的混合情感分类模型
    """
    def __init__(self, bert_model_name='bert-base-uncased', downstream_dim=128, num_heads=4, num_layers=2, dim_feedforward=256, dropout=0.3):
        super(BertTransformerClassifier, self).__init__()

        # 1. 加载预训练 BERT 并冻结全部参数 (仅作为词嵌入提取器)
        self.bert = AutoModel.from_pretrained(bert_model_name)
        for param in self.bert.parameters():
            param.requires_grad = False  # 冻结 BERT 权重

        bert_dim = self.bert.config.hidden_size  # 768 维

        # 2. 将 BERT 的 768 维动态词向量投影到下游手写 Encoder 的维度 (128 维)
        self.embedding_proj = nn.Linear(bert_dim, downstream_dim)

        # 3. 手写 4-Head Transformer Encoder 堆叠
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(downstream_dim, num_heads, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)
        # 4. 全连接二分类头
        self.fc = nn.Linear(downstream_dim, 1)

    def forward(self, input_ids, attention_mask):
        # BERT 提取动态上下文词嵌入 (无需计算梯度)
        with torch.no_grad():
            bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            # 提取最后一层隐藏状态 [batch_size, seq_len, 768]
            contextual_embeddings = bert_outputs.last_hidden_state

        # 线性映射到 128 维送入下游手写 Transformer
        out = self.embedding_proj(contextual_embeddings)  # [batch_size, seq_len, 128]

        # 构造 Padding Mask 用于自注意力层 [batch_size, 1, 1, seq_len]
        mask = attention_mask.unsqueeze(1).unsqueeze(2)

        for layer in self.encoder_layers:
            out = layer(out, mask)

        # Masked Global Max Pooling 掩码全局最大池化 (排除 PAD 影响)
        mask_expanded = attention_mask.unsqueeze(-1).expand_as(out)
        out_masked = out.masked_fill(mask_expanded == 0, -1e9)
        out_pooled = torch.max(out_masked, dim=1)[0]

        out_pooled = self.dropout(out_pooled)
        logits = self.fc(out_pooled).squeeze(1)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertTransformerClassifier(
    bert_model_name='bert-base-uncased',
    downstream_dim=128,
    num_heads=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.3
).to(device)

# 统计可训练参数量 (仅统计我们手写部分，BERT 被冻结)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型初始化完成！运行设备: {device}")
print(f"下游手写网络可训练参数量: {trainable_params:,} (BERT 1.1亿参数已全部冻结作为词嵌入特征层)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


模型初始化完成！运行设备: cuda
下游手写网络可训练参数量: 363,521 (BERT 1.1亿参数已全部冻结作为词嵌入特征层)


## 4. Early Stopping 监控与轻量级下游训练循环

In [ ]:
class EarlyStopping:
    def __init__(self, patience=4, verbose=True, save_path='data/best_transformer_bert_model.pt'):
        self.patience = patience
        self.verbose = verbose
        self.save_path = save_path
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping 计数: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
        torch.save(model.state_dict(), self.save_path)
        if self.verbose:
            print(f"验证集 Loss 改善，已保存最佳权重至 {self.save_path}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0005, weight_decay=1e-4)
early_stopping = EarlyStopping(patience=4, verbose=True, save_path='data/best_transformer_bert_model.pt')

epochs = 15
for epoch in range(1, epochs + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * targets.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == targets).sum().item()
        train_total += targets.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, targets)
            val_loss += loss.item() * targets.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

    early_stopping(epoch_val_loss, model)
    if early_stopping.early_stop:
        print("触发 Early Stopping 条件，终止训练！")
        break

Epoch 01/15 | Train Loss: 0.3709 | Train Acc: 82.67% | Val Loss: 0.2756 | Val Acc: 88.51%
验证集 Loss 改善，已保存最佳权重至 data/best_transformer_bert_model.pt
Epoch 02/15 | Train Loss: 0.2890 | Train Acc: 87.92% | Val Loss: 0.2695 | Val Acc: 89.09%
验证集 Loss 改善，已保存最佳权重至 data/best_transformer_bert_model.pt
Epoch 03/15 | Train Loss: 0.2744 | Train Acc: 88.70% | Val Loss: 0.2856 | Val Acc: 88.80%
EarlyStopping 计数: 1/4
Epoch 04/15 | Train Loss: 0.2687 | Train Acc: 88.73% | Val Loss: 0.2778 | Val Acc: 89.17%
EarlyStopping 计数: 2/4
Epoch 05/15 | Train Loss: 0.2624 | Train Acc: 89.08% | Val Loss: 0.2639 | Val Acc: 89.41%
验证集 Loss 改善，已保存最佳权重至 data/best_transformer_bert_model.pt
Epoch 06/15 | Train Loss: 0.2618 | Train Acc: 89.28% | Val Loss: 0.2990 | Val Acc: 86.91%
EarlyStopping 计数: 1/4
Epoch 07/15 | Train Loss: 0.2544 | Train Acc: 89.42% | Val Loss: 0.2774 | Val Acc: 89.25%
EarlyStopping 计数: 2/4
Epoch 08/15 | Train Loss: 0.2496 | Train Acc: 89.77% | Val Loss: 0.2692 | Val Acc: 89.95%
EarlyStopping 计数: 3/4

## 5. 加载最佳权重并在测试集上进行终极评估

In [ ]:
model.load_state_dict(torch.load('data/best_transformer_bert_model.pt', map_location=device))
model.eval()

test_preds = []
test_probs = []
true_labels = df_test['真实标签'].values

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        test_probs.extend(probs)
        test_preds.extend(preds)

acc = accuracy_score(true_labels, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(true_labels, test_preds, average='binary')
cm = confusion_matrix(true_labels, test_preds)

print("=== 测试集终极评估结果 (BERT 词嵌入 + 手写 4-Head Transformer) ===")
print(f"准确率 (Accuracy) : {acc*100:.2f}%")
print(f"精确率 (Precision): {prec*100:.2f}%")
print(f"召回率 (Recall)   : {rec*100:.2f}%")
print(f"F1 得分 (F1 Score): {f1*100:.2f}%")
print("\n混淆矩阵 (Confusion Matrix):")
print(cm)

=== 测试集终极评估结果 (BERT 词嵌入 + 手写 4-Head Transformer) ===
准确率 (Accuracy) : 89.51%
精确率 (Precision): 90.96%
召回率 (Recall)   : 87.74%
F1 得分 (F1 Score): 89.32%

混淆矩阵 (Confusion Matrix):
[[11410  1090]
 [ 1532 10968]]
